# Bagian 2 — Apple AppStore Analysis
## CRISP-DM Framework

## 1. Business Understanding

**Problem Statement:**
App developers and publishers want to understand what factors drive high user ratings
on the Apple AppStore. By predicting rating categories (Low/Medium/High), we can
help developers prioritize features and help Apple curate quality apps.

**Business Value:** Guides app development decisions, marketing strategies, and
AppStore curation to maximize user satisfaction.

## 2. Data Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('sample-apple.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Data types and missing values
print(df.info())
print('\nMissing values:')
print(df.isnull().sum().sort_values(ascending=False))

In [ ]:
# Basic stats
df[['Average_User_Rating', 'Reviews', 'Price', 'Size_Bytes']].describe()

In [ ]:
# --- Visualizations ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Rating Distribution
df['Average_User_Rating'].hist(bins=20, ax=axes[0,0], color='steelblue', edgecolor='black')
axes[0,0].set_title('Distribution of Average User Rating')
axes[0,0].set_xlabel('Rating')

# 2. Content Rating Distribution
df['Content_Rating'].value_counts().plot(kind='bar', ax=axes[0,1], color='salmon')
axes[0,1].set_title('Content Rating Distribution')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Free vs Paid
df['Free'].value_counts().plot(kind='pie', ax=axes[1,0], autopct='%1.1f%%',
                                colors=['lightgreen','tomato'], labels=['Free','Paid'])
axes[1,0].set_title('Free vs Paid Apps')

# 4. Price vs Rating (only paid apps)
paid = df[df['Price'] > 0]
axes[1,1].scatter(paid['Price'], paid['Average_User_Rating'], alpha=0.4, color='purple')
axes[1,1].set_title('Price vs Average User Rating (Paid Apps)')
axes[1,1].set_xlabel('Price (USD)')
axes[1,1].set_ylabel('Rating')

plt.tight_layout()
plt.savefig('apple_distributions.png', dpi=150)
plt.show()

In [ ]:
# Top genres by average rating
genre_rating = df.groupby('Primary_Genre')['Average_User_Rating'].mean().sort_values(ascending=False).head(15)
genre_rating.plot(kind='barh', figsize=(10, 6), color='teal')
plt.title('Top 15 Genres by Average User Rating')
plt.xlabel('Average Rating')
plt.tight_layout()
plt.savefig('genre_ratings.png', dpi=150)
plt.show()

## 3. Data Preprocessing & Feature Engineering

In [ ]:
# --- Parse Dates ---
df['Released'] = pd.to_datetime(df['Released'], errors='coerce')
df['Updated']  = pd.to_datetime(df['Updated'],  errors='coerce')

reference_date = pd.Timestamp('today')

# --- Feature Engineering ---

# 1. App Age in days (since release)
df['App_Age_Days'] = (reference_date - df['Released']).dt.days

# 2. Days since last update
df['Days_Since_Update'] = (reference_date - df['Updated']).dt.days

# 3. Review Intensity (reviews per day since release)
df['Review_Intensity'] = df['Reviews'] / (df['App_Age_Days'] + 1)

# 4. Rating-to-Review Ratio
df['Rating_Review_Ratio'] = df['Average_User_Rating'] / (df['Reviews'] + 1)

# 5. Size in MB
df['Size_MB'] = df['Size_Bytes'] / 1_000_000

# 6. Is Free (convert boolean to int)
df['Is_Free'] = df['Free'].astype(int)

print(df[['App_Age_Days', 'Days_Since_Update', 'Review_Intensity',
          'Rating_Review_Ratio', 'Size_MB', 'Is_Free']].head())

In [ ]:
# --- Create Target Variable: Rating Category ---
# Apps with 0 rating are likely unrated — handle separately
df_model = df[df['Average_User_Rating'] > 0].copy()

def rating_category(r):
    if r < 3.5:
        return 'Low'
    elif r < 4.5:
        return 'Medium'
    else:
        return 'High'

df_model['Rating_Category'] = df_model['Average_User_Rating'].apply(rating_category)
print(df_model['Rating_Category'].value_counts())

In [ ]:
# --- Encode Categorical Features ---
from sklearn.preprocessing import LabelEncoder

le_genre   = LabelEncoder()
le_content = LabelEncoder()

df_model['Genre_Encoded']   = le_genre.fit_transform(df_model['Primary_Genre'].fillna('Unknown'))
df_model['Content_Encoded'] = le_content.fit_transform(df_model['Content_Rating'].fillna('Unknown'))

# Define feature set
FEATURES = [
    'Size_MB', 'Price', 'Is_Free', 'App_Age_Days', 'Days_Since_Update',
    'Review_Intensity', 'Rating_Review_Ratio', 'Reviews',
    'Genre_Encoded', 'Content_Encoded'
]

X = df_model[FEATURES].fillna(0)
y = df_model['Rating_Category']

print('Feature matrix shape:', X.shape)
print('Class distribution:\n', y.value_counts())

## 4. Data Modelling — Classification

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

In [ ]:
# --- Helper function to train + evaluate ---
def train_evaluate(model, X_tr, X_te, y_tr, y_te, model_name):
    start = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - start

    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)

    print(f'\n=== {model_name} ===')
    print(f'Training time : {train_time:.4f}s')
    print(f'Accuracy      : {acc:.4f}')
    print(classification_report(y_te, y_pred))

    # Confusion matrix
    cm = confusion_matrix(y_te, y_pred, labels=['Low','Medium','High'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Low','Medium','High'],
                yticklabels=['Low','Medium','High'])
    plt.title(f'Confusion Matrix — {model_name}')
    plt.tight_layout()
    plt.savefig(f'cm_{model_name.replace(" ","_")}.png', dpi=150)
    plt.show()

    return acc, train_time

In [ ]:
# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
acc_lr, time_lr = train_evaluate(lr, X_train_scaled, X_test_scaled,
                                  y_train, y_test, 'Logistic Regression')

In [ ]:
# 2. Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
acc_dt, time_dt = train_evaluate(dt, X_train, X_test,
                                  y_train, y_test, 'Decision Tree')

In [ ]:
# 3. Random Forest (ensemble)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
acc_rf, time_rf = train_evaluate(rf, X_train, X_test,
                                  y_train, y_test, 'Random Forest')

## 5. Evaluation — Model Comparison

In [ ]:
# Summary comparison table
results = pd.DataFrame({
    'Model'        : ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy'     : [acc_lr, acc_dt, acc_rf],
    'Training Time': [time_lr, time_dt, time_rf]
})
print(results.to_string(index=False))

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results.plot(x='Model', y='Accuracy',     kind='bar', ax=axes[0], legend=False, color='steelblue')
results.plot(x='Model', y='Training Time',kind='bar', ax=axes[1], legend=False, color='coral')
axes[0].set_title('Model Accuracy')
axes[1].set_title('Training Time (seconds)')
for ax in axes:
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

In [ ]:
# Feature Importance (Random Forest)
feat_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
feat_imp.plot(kind='barh', figsize=(10, 6), color='green')
plt.title('Feature Importance — Random Forest')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

## 6. Limitations & Lesson Learned

- **Limitations:** Ratings with 0 value were excluded (unrated apps); class imbalance may affect results (consider SMOTE or class_weight).
- **Challenges:** Many apps have 0 reviews making Review_Intensity meaningless; price distribution is heavily skewed (most apps are free).
- **Lesson Learned:** Feature engineering (App_Age, Review_Intensity) significantly improves model performance compared to using raw columns alone. Random Forest handles non-linear relationships better than Logistic Regression for this dataset.